# Continuous Control

---

In this notebook, you will learn how to use the Unity ML-Agents environment for the second project of the [Deep Reinforcement Learning Nanodegree](https://www.udacity.com/course/deep-reinforcement-learning-nanodegree--nd893) program.

### 1. Start the Environment

We begin by importing the necessary packages.  If the code cell below returns an error, please revisit the project instructions to double-check that you have installed [Unity ML-Agents](https://github.com/Unity-Technologies/ml-agents/blob/master/docs/Installation.md) and [NumPy](http://www.numpy.org/).

In [ ]:
from unityagents import UnityEnvironment
import numpy as np

Next, we will start the environment!  **_Before running the code cell below_**, change the `file_name` parameter to match the location of the Unity environment that you downloaded.

- **Mac**: `"path/to/Reacher.app"`
- **Windows** (x86): `"path/to/Reacher_Windows_x86/Reacher.exe"`
- **Windows** (x86_64): `"path/to/Reacher_Windows_x86_64/Reacher.exe"`
- **Linux** (x86): `"path/to/Reacher_Linux/Reacher.x86"`
- **Linux** (x86_64): `"path/to/Reacher_Linux/Reacher.x86_64"`
- **Linux** (x86, headless): `"path/to/Reacher_Linux_NoVis/Reacher.x86"`
- **Linux** (x86_64, headless): `"path/to/Reacher_Linux_NoVis/Reacher.x86_64"`

For instance, if you are using a Mac, then you downloaded `Reacher.app`.  If this file is in the same folder as the notebook, then the line below should appear as follows:
```
env = UnityEnvironment(file_name="Reacher.app")
```

In [ ]:
env = UnityEnvironment(file_name='reacher/Reacher.app')

Environments contain **_brains_** which are responsible for deciding the actions of their associated agents. Here we check for the first brain available, and set it as the default brain we will be controlling from Python.

In [ ]:
# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

### 2. Examine the State and Action Spaces

In this environment, a double-jointed arm can move to target locations. A reward of `+0.1` is provided for each step that the agent's hand is in the goal location. Thus, the goal of your agent is to maintain its position at the target location for as many time steps as possible.

The observation space consists of `33` variables corresponding to position, rotation, velocity, and angular velocities of the arm.  Each action is a vector with four numbers, corresponding to torque applicable to two joints.  Every entry in the action vector must be a number between `-1` and `1`.

Run the code cell below to print some information about the environment.

In [ ]:
# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])

### 3. Take Random Actions in the Environment

In the next code cell, you will learn how to use the Python API to control the agent and receive feedback from the environment.

Once this cell is executed, you will watch the agent's performance, if it selects an action at random with each time step.  A window should pop up that allows you to observe the agent, as it moves through the environment.  

Of course, as part of the project, you'll have to change the code so that the agent is able to use its experience to gradually choose better actions when interacting with the environment!

In [ ]:
env_info = env.reset(train_mode=False)[brain_name]     # reset the environment    
states = env_info.vector_observations                  # get the current state (for each agent)
scores = np.zeros(num_agents)                          # initialize the score (for each agent)
while True:
    actions = np.random.randn(num_agents, action_size) # select an action (for each agent)
    actions = np.clip(actions, -1, 1)                  # all actions between -1 and 1
    env_info = env.step(actions)[brain_name]           # send all actions to the environment
    next_states = env_info.vector_observations         # get next state (for each agent)
    rewards = env_info.rewards                         # get reward (for each agent)
    dones = env_info.local_done                        # see if episode finished
    scores += env_info.rewards                         # update the score (for each agent)
    states = next_states                               # roll over states to next time step
    if np.any(dones):                                  # exit loop if episode finished
        break
print('Total score (averaged over agents) this episode: {}'.format(np.mean(scores)))

When finished, you can close the environment.

In [ ]:
env.close()

### 4. Train the Agent with DDPG

Now we train our own agent. We use Deep Deterministic Policy Gradient (DDPG), an actor-critic method for continuous action spaces.

- The **actor** network decides which torques to apply, given the current arm state.
- The **critic** network estimates how good a state-action pair is (the Q-value), and its gradient teaches the actor to pick better actions.
- Experience from all 20 agents is pooled into a shared **replay buffer**, and training samples random minibatches from it. This decorrelates the data and lets every agent learn from the others' experience.
- **Target networks** (slow-moving copies of actor and critic) stabilize the Q-value targets, and **Ornstein-Uhlenbeck noise** provides temporally smooth exploration of the torque space.

When training the environment, set `train_mode=True`, so that the line for resetting the environment looks like the following:
```python
env_info = env.reset(train_mode=True)[brain_name]
```

Checkpoints are saved after every episode. Once the average score over the last 100 episodes reaches +30, the current weights are kept as the best model and training stops.


In [ ]:
from unityagents import UnityEnvironment
import numpy as np
from collections import deque
import torch
from agent import Agent
from matplotlib.pylab import plt

In [ ]:
env = UnityEnvironment(file_name='reacher/Reacher 2.app', base_port=64738, worker_id=2, seed=1, no_graphics=True)

# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])

In [ ]:
def ddpg_multi_agents(agent, n_episodes=350, max_t_steps=2048):
    """Train the agent with DDPG across all parallel agents.

    Each episode, every agent acts in its own copy of the environment and
    all transitions are stored in the agent's shared replay buffer. The
    per-episode score is the mean over agents of their undiscounted returns.
    Training stops early once the rolling 100-episode average reaches +30,
    the score that marks the environment as solved.
    """
    scores_deque = deque(maxlen=100)
    all_scores = []
    for i_episode in range(1, n_episodes + 1):

        env_info = env.reset(train_mode=True)[brain_name]
        states = env_info.vector_observations
        scores = np.zeros(num_agents)
        agent.reset()


        for t in range(max_t_steps):
            actions = agent.act(states)
            env_info = env.step(actions)[brain_name]        # send the action to the environment
            next_states = env_info.vector_observations     # get the next state
            rewards = env_info.rewards                     # get the reward
            dones = env_info.local_done

            agent.step(states, actions, rewards, next_states, dones, t)

            states = next_states
            scores += rewards

            if np.any(dones):
                break

        scores_deque.append(np.mean(scores))
        all_scores.append(np.mean(scores))
        print('\rEpisode {}\t Current Avg Score: {:.2f} \tAverage Score in Queue: {:.2f}'.format(i_episode, np.mean(scores), np.mean(scores_deque)), end="")

        # checkpoint after every episode; keep the weights once the environment is solved
        torch.save(agent.actor_local.state_dict(), 'reacher_checkpoint_actor.pth')
        torch.save(agent.critic_local.state_dict(), 'reacher_checkpoint_critic.pth')
        if np.mean(scores_deque) > 30:
            torch.save(agent.actor_local.state_dict(), 'reacher_best_actor.pth')
            torch.save(agent.critic_local.state_dict(), 'reacher_best_critic.pth')
            break

    return all_scores


In [ ]:
agent_clone = Agent(state_size=state_size, action_size=action_size,num_agents=num_agents, random_seed=100)

scores = ddpg_multi_agents(agent_clone)

# plot the scores
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()

### 5. Watch the Trained Agent

Training is done. Below we load the best saved weights into a fresh agent and let it control the arm with the graphics on, so you can watch the learned reaching behavior. The agent acts greedily here (noise is still added by `act`, but the policy itself is fixed).


In [ ]:
from unityagents import UnityEnvironment
import numpy as np
import torch
from agent import Agent

In [ ]:
env = UnityEnvironment(file_name='reacher/Reacher 2.app', base_port=64738, worker_id=2, seed=1, no_graphics=False)

# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])

In [ ]:
agent = Agent(state_size=state_size, action_size=action_size, num_agents=num_agents, random_seed=100)

# Load the trained weights
agent.critic_local.load_state_dict(torch.load('reacher_best_critic.pth'))
agent.actor_local.load_state_dict(torch.load('reacher_best_actor.pth'))


env_info = env.reset(train_mode=False)[brain_name]
states = env_info.vector_observations
scores = np.zeros(num_agents)

# Run one greedy episode and watch the arm reach for the target
while True:
    actions = agent.act(states)
    env_info = env.step(actions)[brain_name]
    next_states = env_info.vector_observations
    rewards = env_info.rewards
    dones = env_info.local_done
    scores += env_info.rewards
    states = next_states
    if np.any(dones):
        break

env.close()


In [ ]:
# plot the scores
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()